In [ ]:
import pickle
import pandas as pd
import numpy as np
from src.utils import load_env, set_seed, get_logger, load_json
from case_study.utils import set_matplot_style
from src.data_loading import DatasetLoader
from src.experiment_config import ExperimentConfig
from experiments.cluster_lr import load_filtered_raw_data

DATASET = "tweets_immigration"
LABELED_ONLY = True

CLUSTER_TEC = "kmeans" # or pckmeans
K_VAL = 200

# only applies if pckmeans set 
PCK_WEIGHT = 0.05

EMB_NAME = "mets_only_ling_mets"
ABLATION_NAME = None

set_matplot_style()
env_vars = load_env()
logger = get_logger("cluster_lr_eval")
set_seed(env_vars["RANDOM_SEED"])

In [ ]:
# get data loader 
config = ExperimentConfig(
        task_type="cluster_lr",
        task_name="",
        dataset=DATASET,
        labeled_only=LABELED_ONLY,
        logger=logger,
        env_vars=env_vars
    )
data_loader = DatasetLoader(config)

In [ ]:
# load metaphor classification results 
met_class_path = f"{env_vars["RESULTS_DIR"]}/metaphor_classification/llm/{config.dataset_out_name}_source_verb_target_noun_binary_met_class_qwen.json"
met_class_data = load_json(met_class_path)["data"]

In [ ]:
mets = {k: v for k, v in met_class_data.items() if v["annotation"]["classification"] == "Metaphorical"}
doc_ids = [k.split("_")[0] for k in mets.keys()]
unique_doc_ids = list(set(doc_ids))

print(f"{len(mets)}/{len(met_class_data)} candidates labeled 'Metaphorical' from {len(unique_doc_ids)} documents")

In [ ]:
# load metaphor embedding data
# met_emb_data_path = f"{env_vars["RESULTS_DIR"]}/gen_embeddings/sbert/{DATASET}_{EMB_NAME}_features.pkl"
met_emb_data_path = f"{env_vars["RESULTS_DIR"]}/gen_embeddings/sbert/{config.dataset_out_name}_{EMB_NAME}_embeddings.pkl"

with open(met_emb_data_path, "rb") as f:
    met_emb_data = pickle.load(f)

for k, v in met_emb_data.items():
    if 'features' not in v:
        met_emb_data[k]["features"] = {
            "target_group": "", 
            "noun_classification": "",
            "source_image_schema_group": ""            
        }
print(len(met_emb_data))

In [ ]:
# load weights from lr so i can order clusters from most to least important
importance_path = f"{env_vars['RESULTS_DIR']}/cluster_lr/{config.dataset_out_name}/polarity/{CLUSTER_TEC}/{EMB_NAME}/k_{K_VAL}_importance.csv"
if CLUSTER_TEC == "pckmeans":
    importance_path = importance_path.replace("_importance.csv", f"_w{PCK_WEIGHT}_importance.csv")
importance_df = pd.read_csv(importance_path)
importance_df["clust_imp_rank"] = range(1, len(importance_df) + 1)
importance_df = importance_df.set_index('Unnamed: 0')
importance_df.head()


In [ ]:
# load cluster text
met_ent_path = f"{env_vars["RESULTS_DIR"]}/gen_entailments/llm/{config.dataset_out_name}_metaphor_framing_entailments_qwen.json"
met_ent_data = load_json(met_ent_path)["data"]

In [ ]:
met_clust_data_path = f"{env_vars["RESULTS_DIR"]}/cluster_embeddings/{CLUSTER_TEC}/{config.dataset_out_name}/{EMB_NAME}_clusters_{K_VAL}.pkl"
if ABLATION_NAME:
    met_clust_data_path = f"{env_vars["RESULTS_DIR"]}/cluster_ablation/{ABLATION_NAME}"
else: 
    met_clust_data_path = f"{env_vars["RESULTS_DIR"]}/cluster_embeddings/{CLUSTER_TEC}/{config.dataset_out_name}"

clust_file_name = f"{EMB_NAME}_clusters_{K_VAL}.pkl"
if CLUSTER_TEC == "pckmeans":
    clust_file_name = clust_file_name.replace(".pkl", f"_w{PCK_WEIGHT}.pkl")

with open(f"{met_clust_data_path}/{clust_file_name}", "rb") as f:
    met_clust_data = pickle.load(f)

print(met_clust_data.keys())

In [ ]:
print(f"Number of clusters: {met_clust_data["number_cluster"]}")
print(f"Number of embeddings: {len(met_clust_data["embeddings"])}")
print(f"Embedding shape: {met_emb_data[str(0)]['emb'].shape}")

assert len(met_emb_data) == len(met_clust_data['labels'])

In [ ]:
polarity = load_filtered_raw_data(
    dataset=DATASET, labeled_only=LABELED_ONLY, label_list=['polarity'], 
    filter_to_met_docs=False, logger=logger, data_loader=data_loader
)

In [ ]:
print(len(polarity))
print(len(met_clust_data['labels']))

doc_ids = [met_emb_data[str(met_idx)]['id'].split("_")[0] for met_idx, clust_label in enumerate(met_clust_data['labels'])]
unique_doc_ids = list(set(doc_ids))
print(len(unique_doc_ids))

In [ ]:
# make a dict {id: {cluster_label: '', text: ''}}
clean_clust_dict = {
    met_emb_data[str(met_idx)]['id']: {
        'met_id': met_emb_data[str(met_idx)]['id'],
        'met_idx': str(met_idx),
        'cluster': clust_label,
        'clust_imp_rank': importance_df.loc[clust_label, 'clust_imp_rank'],
        'doc_id': met_emb_data[str(met_idx)]['id'].split("_")[0],
        'polarity': polarity[met_emb_data[str(met_idx)]['id'].split("_")[0]]['polarity'],
        'target_group': met_emb_data[str(met_idx)]['features']['target_group'],
        'target_ppto': met_emb_data[str(met_idx)]['features']['noun_classification'],
        'source_image_schema_group': met_emb_data[str(met_idx)]['features']['source_image_schema_group'],
        'source_word': met_class_data[met_emb_data[str(met_idx)]['id']]['source_word'], 
        'target_word': met_class_data[met_emb_data[str(met_idx)]['id']]['target_word'],
        "entailment_text": (" ").join(met_ent_data[met_emb_data[str(met_idx)]['id']]['annotation'].values()),
        'sentence': met_class_data[met_emb_data[str(met_idx)]['id']]['text'].split("Sentence: ")[1],
        'distance': np.linalg.norm(met_emb_data[str(met_idx)]['emb'] - met_clust_data["cluster_centers"][clust_label])

    } for met_idx, clust_label in enumerate(met_clust_data['labels'])
}

In [ ]:
met_emb_data[str(0)]['features']['target_group']

In [ ]:
# # load raw data for polarity labels
# config = ExperimentConfig(
#     None, None, "podcasts", None, env_vars, skip_load=True,
# )
# loader = DatasetLoader(config)
# polarity_df = loader.load_raw_data()
# print(len(polarity_df))

In [ ]:
file_name = f"{DATASET}_{CLUSTER_TEC}_k{K_VAL}.csv"
if CLUSTER_TEC == "pckmeans":
    file_name = file_name.replace(".csv", f"_w{PCK_WEIGHT}.csv")

clean_clust_df = pd.DataFrame.from_dict(clean_clust_dict, orient="index")
clean_clust_df["target_group"] = clean_clust_df["target_group"].apply(lambda x: x.split(":")[0])

# clean_clust_df = clean_clust_df.merge(polarity_df, left_on="doc_id", right_on="id")
# clean_clust_df = clean_clust_df[["met_id", "cluster", "distance", "polarity", "target_group", "target_ppto", "target_word", "source_image_schema_group", "source_word", "llm_expl", "sentence"]]
clean_clust_df = clean_clust_df[["met_id", "distance", "cluster", "clust_imp_rank", "polarity", "target_group", "target_ppto", "target_word", "source_image_schema_group", "source_word", "sentence", "entailment_text"]]

# clean_clust_df = clean_clust_df.sort_values(by=['cluster', 'target_group', 'source_image_schema_group'], ascending=True)
clean_clust_df = clean_clust_df.sort_values(by=['clust_imp_rank', 'distance'], ascending=True)

# clean_clust_df.to_csv(f"clusters/{file_name}")
csv_name = clust_file_name.replace(".pkl", ".csv")
out_path = f"{met_clust_data_path}/{csv_name}"

clean_clust_df.to_csv(out_path)
print(f"Clusters saved to: {out_path}")

In [ ]:
# Keep only the closest 25% of instances to the centroid within each cluster
closest_df = (
    clean_clust_df
    .groupby("cluster", group_keys=False)
    .apply(lambda g: g.nsmallest(max(1, int(np.ceil(len(g) * 0.25))), "distance"))
    .sort_values(["clust_imp_rank", "distance"], ascending=[True, True])
    .reset_index(drop=True)
)

closest_csv_name = csv_name.replace(".csv", "_closest25.csv")
closest_df.to_csv(f"{met_clust_data_path}/{closest_csv_name}", index=False)
print(f"Closest 25% per cluster saved to: {met_clust_data_path}/{closest_csv_name}")
